# 01 - LCEL 核心基础

## 学习目标

1. 理解 LCEL 的 `|` 管道操作符及其数据流模型
2. 掌握 `RunnablePassthrough`：透传输入、使用 `assign()` 添加字段
3. 掌握 `RunnableLambda`：将任意 Python 函数包装为可运行组件
4. 掌握 `RunnableParallel`：并发执行多条链并合并输出
5. 掌握 `RunnableBranch`：基于输入的条件路由
6. 掌握 `RunnableSequence`：显式链式调用
7. 构建最终综合示例：`prompt | model | parser`

## LCEL 核心理念

LangChain Expression Language (LCEL) 是一种声明式语言，用于组合 LangChain 组件。
其核心思想是：**链中的每个组件都是 "Runnable"（可运行的），数据通过 `|` 操作符从左向右流动**。

```
输入 → component_a | component_b | component_c → 输出
```

每个组件的 `.invoke()` 方法接收输入，产生输出，输出自动成为下一个组件的输入。

In [ ]:
# 基础导入
import sys
sys.path.insert(0, '../..')

from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda,
    RunnableParallel,
    RunnableBranch,
    RunnableSequence,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig

# 用于 Mock LLM 的辅助函数
from typing import Any, Dict

print("All imports successful.")

---

## 1. 管道操作符 `|`

### 概念说明

`|` 操作符是 LCEL 的核心语法。它将两个 Runnable 对象连接在一起，
前一个的输出自动成为后一个的输入。LangChain 通过重载 Python 的 `__or__` 方法实现此功能。

```
chain = runnable_a | runnable_b
result = chain.invoke(input)  # 等价于 runnable_b.invoke(runnable_a.invoke(input))
```

### Mock LLM

为了演示目的，我们创建一个 Mock LLM 来模拟 LLM 的行为，
避免实际 API 调用。

In [ ]:
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langchain_core.outputs import ChatResult, ChatGeneration


class MockChatModel(BaseChatModel):
    """一个简单的 Mock ChatModel，用于演示 LCEL 管道。
    
    将输入消息反转后返回，以展示数据在链中的流动。
    """
    
    def _generate(
        self,
        messages: list[BaseMessage],
        stop: list[str] | None = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> ChatResult:
        last_message = messages[-1]
        content = last_message.content if hasattr(last_message, 'content') else str(last_message)
        response_text = f"[Mock LLM Response to: '{content[:50]}...']"
        message = AIMessage(content=response_text)
        generation = ChatGeneration(message=message)
        return ChatResult(generations=[generation])
    
    @property
    def _llm_type(self) -> str:
        return "mock-chat-model"


mock_model = MockChatModel()
print(f"Created MockChatModel: type={type(mock_model).__name__}")
print(f"Is Runnable: {hasattr(mock_model, 'invoke')}")

### 示例 1.1：最简单的管道

将两个转换函数串联起来。注意数据如何从一个组件流向下一个组件。

In [ ]:
# 定义两个简单的 RunnableLambda 作为管道组件
step1 = RunnableLambda(lambda x: x.upper())
step2 = RunnableLambda(lambda x: f"结果: {x}")

# 使用 | 操作符连接
simple_chain = step1 | step2

# 测试
result = simple_chain.invoke("hello lcel world")
print(f"输入: 'hello lcel world'")
print(f"输出: {result}")

# 分步验证数据流
print("\n--- 分步验证 ---")
intermediate = step1.invoke("hello lcel world")
print(f"Step1 输出: {intermediate}")
final = step2.invoke(intermediate)
print(f"Step2 输出: {final}")
print(f"管道invoke结果: {simple_chain.invoke('hello lcel world')}")

### 示例 1.2：多步数据转换管道

展示一个包含多个步骤的实际转换管道。

In [ ]:
def remove_punctuation(text: str) -> str:
    """移除文本中的标点符号。"""
    import re
    return re.sub(r'[^\w\s]', '', text)


def tokenize(text: str) -> list[str]:
    """将文本拆分为单词列表。"""
    return text.split()


def count_tokens(tokens: list[str]) -> dict:
    """统计单词数量并返回分析结果。"""
    return {
        "token_count": len(tokens),
        "unique_tokens": len(set(tokens)),
        "tokens": tokens
    }


# 构建多步分析管道
analysis_chain = (
    RunnableLambda(remove_punctuation)
    | RunnableLambda(tokenize)
    | RunnableLambda(count_tokens)
)

# 测试
sample_text = "Hello, World! This is a test. Hello, LCEL!"
result = analysis_chain.invoke(sample_text)
print(f"输入: {sample_text}")
print(f"分析结果: {result}")

---

## 2. RunnablePassthrough

### 概念说明

`RunnablePassthrough` 是一个特殊的 Runnable，它将输入原封不动地传递给输出。
这在以下场景中非常有用：

- **数据占位**：在链中传递数据而不做修改
- **`assign()` 方法**：在传递原有数据的同时添加新字段
- **构建复合输入**：为下游组件组装多个输入源

```python
# 基础用法：输入 = 输出
passthrough = RunnablePassthrough()
result = passthrough.invoke({"name": "Alice"})  # 返回 {"name": "Alice"}
```

### 示例 2.1：基础透传

验证 RunnablePassthrough 的基本行为。

In [ ]:
# 基础透传：输入即输出
passthrough = RunnablePassthrough()

test_inputs = [
    "simple string",
    {"key": "value"},
    [1, 2, 3],
    42,
]

for inp in test_inputs:
    result = passthrough.invoke(inp)
    input_type = type(inp).__name__
    print(f"输入类型: {input_type:15s} | 输入: {str(inp):30s} | 相同: {result == inp}")

### 示例 2.2：使用 `assign()` 添加字段

`assign()` 是 `RunnablePassthrough` 最强大的功能之一。
它接收一个键值对字典，其中值可以是 Runnable 对象，
在保持原有字段不变的同时，计算并添加新字段。

这在 RAG 链中特别有用——我们需要保留原始问题，同时添加检索到的上下文。

In [ ]:
# 场景：处理用户数据，添加计算字段
user_enricher = RunnablePassthrough.assign(
    # 添加全名字段
    full_name=lambda x: f"{x['first_name']} {x['last_name']}",
    # 添加大写姓名字段
    upper_name=lambda x: x['first_name'].upper() + " " + x['last_name'].upper(),
    # 添加域名建议
    domain_suggestion=lambda x: f"{x['first_name'].lower()}{x['last_name'].lower()}.com",
)

test_user = {
    "first_name": "Zhang",
    "last_name": "San",
    "age": 28,
    "city": "Beijing"
}

enriched_user = user_enricher.invoke(test_user)
print("原始数据:")
for k, v in test_user.items():
    print(f"  {k}: {v}")
print("\n增强后数据（所有原有字段已保留）:")
for k, v in enriched_user.items():
    print(f"  {k}: {v}")

### 示例 2.3：在链中使用 assign() 组装 RAG 输入

模拟 RAG 链中的典型模式：保留用户问题，同时获取检索上下文。

In [ ]:
# 模拟检索器
def mock_retriever(query: str) -> str:
    """模拟向量检索，返回相关文档。"""
    return f"[检索到的文档内容与 '{query}' 相关]"


# 模拟格式化函数
def format_docs(docs: str) -> str:
    """将检索文档格式化为提示上下文。"""
    return f"上下文: {docs}"


# 构建 RAG 输入组装链
rag_input_builder = RunnablePassthrough.assign(
    # context 字段由检索管道生成
    context=RunnableLambda(mock_retriever) | RunnableLambda(format_docs),
    # question 字段保持原始输入（也可显式提取）
    # 原始 question 会由 RunnablePassthrough 自动保留
)

# 测试
query_input = {"question": "什么是 LCEL？"}
built_input = rag_input_builder.invoke(query_input)

print("原始输入:")
for k, v in query_input.items():
    print(f"  {k}: {v}")
print("\n组装后的输入（新增了 context 字段）:")
for k, v in built_input.items():
    print(f"  {k}: {v}")

---

## 3. RunnableLambda

### 概念说明

`RunnableLambda` 将任意 Python 函数（或 lambda）包装为一个 Runnable 对象。
这使得我们可以将自定义逻辑无缝集成到 LCEL 管道中。

```python
runnable = RunnableLambda(my_function)
result = runnable.invoke(input_data)
```

关键特性：
- 自动支持同步和异步函数
- 可以访问 RunnableConfig（用于回调、标签等）
- 输入类型会自动推断

### 示例 3.1：包装各种类型的函数

In [ ]:
# 1. 包装简单函数
runnable_double = RunnableLambda(lambda x: x * 2)
print(f"RunnableLambda(lambda x: x * 2).invoke(21) = {runnable_double.invoke(21)}")

# 2. 包装命名函数
def add_exclamation(text: str) -> str:
    """在文本末尾添加感叹号。"""
    return f"{text}!"

runnable_exclaim = RunnableLambda(add_exclamation)
print(f"RunnableLambda(add_exclamation).invoke('Hello') = {runnable_exclaim.invoke('Hello')}")

# 3. 包装接收字典并返回字典的函数
def process_user(user_info: dict) -> dict:
    """处理用户信息，返回增强后的字典。"""
    return {
        **user_info,
        "greeting": f"你好, {user_info.get('name', '用户')}!",
        "is_vip": user_info.get('level', 0) >= 5
    }

runnable_user_processor = RunnableLambda(process_user)
test_user = {"name": "小明", "level": 7}
result = runnable_user_processor.invoke(test_user)
print(f"\n用户处理结果:")
for k, v in result.items():
    print(f"  {k}: {v}")

### 示例 3.2：访问 RunnableConfig

RunnableLambda 允许函数接收第二个参数 `config`，用于访问运行时配置。

In [ ]:
def function_with_config(input_data: str, config: RunnableConfig) -> str:
    """演示如何在 RunnableLambda 中访问配置。"""
    tags = config.get("tags", [])
    metadata = config.get("metadata", {})
    run_name = metadata.get("run_name", "未知")
    return f"[{run_name}] 处理 '{input_data}' (tags: {tags})"


runnable_with_config = RunnableLambda(function_with_config)

# 不带配置调用
result1 = runnable_with_config.invoke("测试数据")
print(f"无配置: {result1}")

# 带配置调用
result2 = runnable_with_config.invoke(
    "测试数据",
    config=RunnableConfig(
        tags=["demo", "lcel"],
        metadata={"run_name": "lcel_demo"}
    )
)
print(f"有配置: {result2}")

### 示例 3.3：RunnableLambda 组成的链条

In [ ]:
# 构建一个文本处理管道
text_processing_pipeline = (
    RunnableLambda(lambda text: text.strip())           # 去除首尾空格
    | RunnableLambda(lambda text: text.lower())          # 转小写
    | RunnableLambda(lambda text: text.replace("  ", " "))  # 合并多余空格
    | RunnableLambda(lambda text: f"处理后: [{text}]")
)

sample_text = "   HELLO    LCEL   World   "
result = text_processing_pipeline.invoke(sample_text)
print(f"原始: '{sample_text}'")
print(f"结果: {result}")

---

## 4. RunnableParallel

### 概念说明

`RunnableParallel` 接收一个字典，其中每个值都是一个 Runnable。
所有子 Runnable 会**并发执行**，各自的输出合并为一个字典。

```python
parallel_chain = RunnableParallel(
    result_a=chain_a,  # 并发执行
    result_b=chain_b,  # 并发执行
)
output = parallel_chain.invoke(input)
# output = {"result_a": <output_of_chain_a>, "result_b": <output_of_chain_b>}
```

这在需要从同一输入派生多个结果的场景中非常高效。

### 示例 4.1：基础并行执行

对同一输入执行多种分析，汇总结果。

In [ ]:
import time

# 模拟耗时操作
def analyze_sentiment(text: str) -> dict:
    """模拟情感分析。"""
    time.sleep(0.1)  # 模拟API延迟
    positive_words = ["好", "棒", "优秀", "喜欢"]
    score = sum(1 for w in positive_words if w in text) / max(len(text), 1) * 100
    return {"sentiment_score": round(score, 2), "label": "正向" if score > 0.5 else "中性"}


def extract_keywords(text: str) -> dict:
    """模拟关键词提取。"""
    time.sleep(0.1)  # 模拟API延迟
    words = text.replace("，", " ").replace("。", "").split()
    return {"keywords": words[:5] if len(words) >= 5 else words}


def count_statistics(text: str) -> dict:
    """文本统计。"""
    return {
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": text.count("\n") + 1
    }


# 创建并行分析链
parallel_analysis = RunnableParallel(
    sentiment=RunnableLambda(analyze_sentiment),
    keywords=RunnableLambda(extract_keywords),
    statistics=RunnableLambda(count_statistics),
)

# 测试
test_text = "这个产品非常好用，我很喜欢它的设计，功能也很优秀。"
print(f"分析文本: {test_text}")
print()

start_time = time.time()
results = parallel_analysis.invoke(test_text)
elapsed = time.time() - start_time

print(f"并行分析结果 (耗时: {elapsed:.4f}秒):")
for analysis_name, analysis_result in results.items():
    print(f"  [{analysis_name}] {analysis_result}")

### 示例 4.2：并行 vs 串行性能对比

对比并行执行与串行执行的时间差异。

In [ ]:
def slow_operation_a(x):
    time.sleep(0.15)
    return f"A处理了{x}"

def slow_operation_b(x):
    time.sleep(0.15)
    return f"B处理了{x}"

def slow_operation_c(x):
    time.sleep(0.15)
    return f"C处理了{x}"


# 串行方式
serial_a = RunnableLambda(slow_operation_a)
serial_b = RunnableLambda(slow_operation_b)
serial_c = RunnableLambda(slow_operation_c)

start_serial = time.time()
result_a = serial_a.invoke("data")
result_b = serial_b.invoke("data")
result_c = serial_c.invoke("data")
serial_time = time.time() - start_serial

serial_result = {"a": result_a, "b": result_b, "c": result_c}

# 并行方式
parallel = RunnableParallel(
    a=RunnableLambda(slow_operation_a),
    b=RunnableLambda(slow_operation_b),
    c=RunnableLambda(slow_operation_c),
)

start_parallel = time.time()
parallel_result = parallel.invoke("data")
parallel_time = time.time() - start_parallel

print("=== 性能对比 ===")
print(f"串行耗时: {serial_time:.4f}秒")
print(f"并行耗时: {parallel_time:.4f}秒")
print(f"加速比:   {serial_time/parallel_time:.2f}x")
print(f"\n串行结果: {serial_result}")
print(f"并行结果: {parallel_result}")
print(f"\n结果一致: {serial_result == parallel_result}")

---

## 5. RunnableBranch

### 概念说明

`RunnableBranch` 实现条件路由：根据输入匹配条件，选择对应的分支执行。
类似于编程语言中的 `if/elif/else` 语句，但以声明式方式表达。

```python
branch = RunnableBranch(
    (condition_1, handler_1),   # if condition_1 → handler_1
    (condition_2, handler_2),   # elif condition_2 → handler_2
    default_handler,            # else → default_handler
)
```

条件函数接收输入并返回 `True` 或 `False`。

### 示例 5.1：基于类型的分支路由

In [ ]:
# 定义条件函数
def is_greeting(text: str) -> bool:
    """判断是否为问候语。"""
    greetings = ["你好", "hello", "hi", "您好", "早上好"]
    return any(g in text.lower() for g in greetings)


def is_question(text: str) -> bool:
    """判断是否为问题。"""
    return "?" in text or "？" in text or "什么" in text or "怎么" in text


def is_farewell(text: str) -> bool:
    """判断是否为告别语。"""
    farewells = ["再见", "bye", "拜拜", "下次见"]
    return any(f in text.lower() for f in farewells)


# 定义处理函数
greeting_handler = RunnableLambda(lambda x: f"[问候响应] 您好！有什么可以帮您的吗？")
question_handler = RunnableLambda(lambda x: f"[问答响应] 关于'{x}'，这是一个很好的问题...")
farewell_handler = RunnableLambda(lambda x: f"[告别响应] 再见！祝您有美好的一天！")
default_handler = RunnableLambda(lambda x: f"[默认响应] 我不太理解'{x}'，请重新表述。")

# 构建分支路由
router = RunnableBranch(
    (is_greeting, greeting_handler),
    (is_question, question_handler),
    (is_farewell, farewell_handler),
    default_handler,  # 默认分支（else）
)

# 测试各种输入
test_messages = [
    "你好",
    "什么是RAG？",
    "再见！",
    "今天天气不错",
    "您好，请问怎么使用？",
]

for msg in test_messages:
    response = router.invoke(msg)
    print(f"输入: {msg:30s} → 输出: {response}")

### 示例 5.2：数值范围路由

根据输入的数值范围选择不同的处理策略。

In [ ]:
def is_low(score: dict) -> bool:
    return score.get("value", 0) < 60

def is_medium(score: dict) -> bool:
    return 60 <= score.get("value", 0) < 80

def is_high(score: dict) -> bool:
    return score.get("value", 0) >= 80


low_handler = RunnableLambda(lambda x: {
    **x,
    "grade": "不及格",
    "suggestion": "需要加强学习，建议每天额外练习2小时"
})

medium_handler = RunnableLambda(lambda x: {
    **x,
    "grade": "良好",
    "suggestion": "基础扎实，可挑战进阶内容"
})

high_handler = RunnableLambda(lambda x: {
    **x,
    "grade": "优秀",
    "suggestion": "已达到精通水平，可以开始教学或贡献开源项目"
})


score_router = RunnableBranch(
    (is_low, low_handler),
    (is_medium, medium_handler),
    (is_high, high_handler),
)

# 测试
scores = [
    {"name": "Alice", "value": 45},
    {"name": "Bob", "value": 72},
    {"name": "Charlie", "value": 95},
]

for score in scores:
    result = score_router.invoke(score)
    print(f"{result['name']}: 分数={result['value']}, 等级={result['grade']}")
    print(f"  建议: {result['suggestion']}")
    print()

---

## 6. RunnableSequence

### 概念说明

`RunnableSequence` 是 `|` 操作符的显式形式。
当你使用 `a | b | c` 时，LangChain 内部会创建一个 `RunnableSequence`。

在大多数情况下，使用 `|` 操作符更简洁。但 `RunnableSequence` 在以下场景有用：
- 需要程序化地构建序列（如循环添加步骤）
- 需要明确序列的类型以进行类型检查
- 从列表动态构建序列

In [ ]:
from langchain_core.runnables import RunnableSequence

# 方式1：使用 | 操作符（推荐）
chain_pipe = RunnableLambda(lambda x: x * 2) | RunnableLambda(lambda x: x + 10)

# 方式2：使用 RunnableSequence（显式）
chain_explicit = RunnableSequence(
    RunnableLambda(lambda x: x * 2),
    RunnableLambda(lambda x: x + 10),
)

# 方式3：从列表构建（程序化）
steps = [
    RunnableLambda(lambda x: x * 2),
    RunnableLambda(lambda x: x + 10),
]
chain_from_list = RunnableSequence(*steps)

# 验证三种方式等价
test_input = 5
result_pipe = chain_pipe.invoke(test_input)
result_explicit = chain_explicit.invoke(test_input)
result_list = chain_from_list.invoke(test_input)

print(f"输入: {test_input}")
print(f"管道方式:  {result_pipe}")
print(f"显式方式:  {result_explicit}")
print(f"列表方式:  {result_list}")
print(f"三者一致:  {result_pipe == result_explicit == result_list}")

### 示例 6.1：程序化构建序列

当你需要根据条件动态添加步骤时，`RunnableSequence` 非常有用。

In [ ]:
def build_text_pipeline(operations: list[str]) -> RunnableSequence:
    """根据操作列表动态构建文本处理管道。
    
    Args:
        operations: 操作名称列表，可选: 'strip', 'lower', 'upper', 'reverse', 'capitalize'
    
    Returns:
        组合后的 RunnableSequence
    """
    steps = []
    operation_map = {
        "strip": RunnableLambda(lambda t: t.strip()),
        "lower": RunnableLambda(lambda t: t.lower()),
        "upper": RunnableLambda(lambda t: t.upper()),
        "reverse": RunnableLambda(lambda t: t[::-1]),
        "capitalize": RunnableLambda(lambda t: t.capitalize()),
    }
    
    for op in operations:
        if op in operation_map:
            steps.append(operation_map[op])
        else:
            print(f"警告: 未知操作 '{op}'，已跳过")
    
    if not steps:
        return RunnableSequence(RunnablePassthrough())
    
    return RunnableSequence(*steps)


# 测试动态管道构建
test_text = "  Hello LCEL World  "

print(f"原始文本: '{test_text}'")
print()

# 配置1：只需去除空格和转大写
pipeline1 = build_text_pipeline(["strip", "upper"])
print(f"管道1 (strip → upper): '{pipeline1.invoke(test_text)}'")

# 配置2：反转后去掉空格
pipeline2 = build_text_pipeline(["reverse", "strip"])
print(f"管道2 (reverse → strip): '{pipeline2.invoke(test_text)}'")

# 配置3：所有操作
pipeline3 = build_text_pipeline(["strip", "lower", "capitalize"])
print(f"管道3 (strip → lower → capitalize): '{pipeline3.invoke(test_text)}'")

---

## 7. 综合示例：prompt | model | parser

### 概念说明

这是 LCEL 中最经典的组合模式：

```
chain = prompt | model | output_parser
```

1. **prompt**：将输入字典格式化为消息列表
2. **model**：将消息列表发送给 LLM，获得 AI 消息
3. **output_parser**：从 AI 消息中提取结构化内容

这个三件套组合是大多数 LLM 应用的基础。

### 示例 7.1：基础 LLM 链

使用 MockChatModel 展示完整的 `prompt | model | parser` 模式。

In [ ]:
# 步骤1：创建提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有用的AI助手，请用中文回答所有问题。"),
    ("human", "{question}"),
])

# 步骤2：获取模型（使用 MockChatModel 避免API调用）
model = MockChatModel()

# 步骤3：创建输出解析器
parser = StrOutputParser()

# 步骤4：使用 | 构建链
basic_chain = prompt | model | parser

# 步骤5：调用链
result = basic_chain.invoke({"question": "什么是LangChain？"})
print(f"链输出: {result}")

# 验证链的结构
print(f"\n链的类型: {type(basic_chain).__name__}")
print(f"第一步的类型: {type(basic_chain.first).__name__}")
print(f"中间步骤数: {len(basic_chain.middle) if hasattr(basic_chain, 'middle') else 'N/A'}")
print(f"最后一步的类型: {type(basic_chain.last).__name__}")

### 示例 7.2：带预处理的完整链

添加输入预处理步骤，展示更复杂的链组合。

In [ ]:
# 预处理函数：清理和标准化用户输入
def preprocess_input(user_input: dict) -> dict:
    """预处理用户输入，清理问题文本。"""
    question = user_input.get("question", "")
    # 去除多余空格
    question = " ".join(question.split())
    # 确保以问号结尾（如果是问题性质）
    if any(kw in question for kw in ["什么", "怎么", "如何", "为什么"]) and not question.endswith("?") and not question.endswith("？"):
        question += "？"
    return {
        **user_input,
        "question": question,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }


# 后处理函数：格式化最终输出
def postprocess_output(output: str) -> str:
    """后处理模型输出。"""
    separator = "=" * 50
    return f"{separator}\n回答:\n{output}\n{separator}"


# 构建完整链
full_chain = (
    RunnableLambda(preprocess_input)  # 预处理
    | prompt                            # 格式化提示
    | model                             # LLM推理
    | parser                            # 解析输出
    | RunnableLambda(postprocess_output)  # 后处理
)

# 测试
test_question = {"question": "什么是LCEL"}
result = full_chain.invoke(test_question)
print(result)

### 示例 7.3：并行处理 + LLM 链

结合 `RunnableParallel` 和 LLM 链，同时处理多个查询。

In [ ]:
# 定义多个专业领域的提示模板
tech_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个技术专家，请从技术角度回答问题。"),
    ("human", "{question}"),
])

business_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个商业分析师，请从商业角度回答问题。"),
    ("human", "{question}"),
])


# 创建两条独立的链
tech_chain = tech_prompt | MockChatModel() | StrOutputParser()
business_chain = business_prompt | MockChatModel() | StrOutputParser()

# 并行执行：同一个问题从两个角度回答
multi_perspective_chain = RunnableParallel(
    technical_analysis=tech_chain,
    business_analysis=business_chain,
)

# 测试
question = {"question": "LangChain 对企业开发有什么影响？"}
results = multi_perspective_chain.invoke(question)

for perspective, answer in results.items():
    print(f"=== {perspective} ===")
    print(answer[:100] + "..." if len(answer) > 100 else answer)
    print()

---

## 8. 总结与练习

### 核心概念回顾

| 组件 | 用途 | 关键方法 |
|------|------|----------|
| `RunnablePassthrough` | 透传数据，添加字段 | `assign(key=runnable)` |
| `RunnableLambda` | 包装Python函数 | 直接调用 `RunnableLambda(fn)` |
| `RunnableParallel` | 并发执行多条链 | `dict(name=chain)` |
| `RunnableBranch` | 条件路由 | `(condition, handler)` 对 |
| `RunnableSequence` | 显式序列 | `RunnableSequence(*steps)` |
| `\|` 操作符 | 连接组件 | `a \| b \| c` |

### 自我检测

请尝试完成以下练习：

1. 使用 `RunnablePassthrough.assign()` 为一个用户字典添加 `adult` 字段（根据 age >= 18 判断）
2. 使用 `RunnableParallel` 同时执行三个不同的文本分析函数
3. 使用 `RunnableBranch` 构建一个简单的客服路由系统
4. 构建一个完整的 `prompt | model | parser` 链，包含输入验证和输出格式化

In [ ]:
# 练习1：使用 assign 添加 adult 字段
age_check_chain = RunnablePassthrough.assign(
    adult=lambda x: x.get("age", 0) >= 18
)

users = [
    {"name": "Alice", "age": 15},
    {"name": "Bob", "age": 22},
    {"name": "Charlie", "age": 17},
    {"name": "Diana", "age": 30},
]

print("=== 练习1：年龄检查 ===")
for user in users:
    result = age_check_chain.invoke(user)
    print(f"{result['name']}: age={result['age']}, adult={result['adult']}")

In [ ]:
# 练习2：并行文本分析
text_analysis = RunnableParallel(
    char_count=lambda text: len(text),
    word_count=lambda text: len(text.split()),
    is_chinese=lambda text: any('\u4e00' <= c <= '\u9fff' for c in text),
)

print("=== 练习2：并行文本分析 ===")
test_texts = [
    "Hello World from LCEL",
    "你好，世界！这是中文文本分析。",
    "Mixed 混合 text 文本",
]
for text in test_texts:
    result = text_analysis.invoke(text)
    print(f"'{text[:30]}...' → {result}")

In [ ]:
# 练习3：客服路由系统
def is_complaint(text: str) -> bool:
    return any(w in text for w in ["投诉", "不满意", "退款", "差", "坏"])

def is_technical(text: str) -> bool:
    return any(w in text for w in ["错误", "bug", "报错", "不能用", "安装"])

def is_order(text: str) -> bool:
    return any(w in text for w in ["订单", "发货", "物流", "收货", "退换"])


service_router = RunnableBranch(
    (is_complaint, RunnableLambda(lambda x: f"[投诉部门] 我们已收到您的投诉，客服将在24小时内联系您。")),
    (is_technical, RunnableLambda(lambda x: f"[技术支持] 请描述具体的错误信息，我们将尽快解决。")),
    (is_order, RunnableLambda(lambda x: f"[订单部门] 您可以通过订单号查询物流状态。")),
    RunnableLambda(lambda x: f"[综合客服] 请详细描述您的问题，我们会为您找到对应的解决方案。"),
)

print("=== 练习3：客服路由系统 ===")
queries = [
    "我要投诉产品质量",
    "软件安装时报错了",
    "我的订单什么时候发货",
    "我想咨询一下",
]
for q in queries:
    print(f"客户: {q:25s} → {service_router.invoke(q)}")

In [ ]:
# 练习4：完整的 prompt | model | parser 链（含验证）

def validate_input(input_data: dict) -> dict:
    """验证输入数据。"""
    question = input_data.get("question", "")
    if not question or not question.strip():
        raise ValueError("问题不能为空")
    if len(question) < 3:
        raise ValueError(f"问题太短（最少3个字符，当前{len(question)}个）")
    return input_data


def format_output(text: str) -> str:
    """格式化输出。"""
    lines = [
        "╔" + "═" * 48 + "╗",
        "║  AI 回答" + " " * 38 + "║",
        "╠" + "═" * 48 + "╣",
    ]
    # 分行显示，每行最多46个字符
    for i in range(0, len(text), 46):
        chunk = text[i:i+46]
        lines.append(f"║  {chunk:<46s}║")
    lines.append("╚" + "═" * 48 + "╝")
    return "\n".join(lines)


# 构建完整链
final_chain = (
    RunnableLambda(validate_input)
    | ChatPromptTemplate.from_messages([
        ("system", "你是一个简洁的AI助手，用中文回答。"),
        ("human", "{question}"),
    ])
    | MockChatModel()
    | StrOutputParser()
    | RunnableLambda(format_output)
)

print("=== 练习4：完整 prompt | model | parser 链 ===")
try:
    result = final_chain.invoke({"question": "LCEL是什么？"})
    print(result)
except ValueError as e:
    print(f"验证错误: {e}")

---

## 本课小结

在本课中，你学习了 LCEL 的核心构建块：

1. **`|` 管道操作符**：声明式地连接组件，数据从左向右流动
2. **`RunnablePassthrough`**：透传数据，`assign()` 方法用于添加新字段
3. **`RunnableLambda`**：无缝集成 Python 函数到 LCEL 管道
4. **`RunnableParallel`**：并发执行多条链，提升性能
5. **`RunnableBranch`**：基于条件的路由决策
6. **`RunnableSequence`**：程序化地构建序列链

这些基础组件是构建复杂 RAG 链的积木。在下一课中，我们将学习如何构建复杂的提示模板。